# Notebook 04 — Train Our First Model

**Goal:** Split data, encode text, train a model, evaluate it, and save it.

**Pipeline steps covered:**
1. Train/test split
2. Choose model
3. Train
4. Evaluate
5. Save model
6. Predict (inference)

**Input:** `data/processed/matches_ready_for_model.csv`

# Step 1 — Load Modeling Data

This file has **features + target only** — no scores (no data leakage).

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "matches_ready_for_model.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "match_predictor.joblib"

df = pd.read_csv(DATA_PATH)

FEATURE_COLUMNS = ["home_team", "away_team", "tournament", "is_neutral", "year", "month"]
TARGET_COLUMN = "home_result"

print("Rows:", len(df))
print("Target distribution:")
print(df[TARGET_COLUMN].value_counts())

Rows: 5711
Target distribution:
home_result
Win     2741
Loss    1662
Draw    1308
Name: count, dtype: int64


# Step 2 — Train/Test Split

**Train/test split** = divide data into two groups:

| Set | Purpose | Analogy |
|-----|---------|--------|
| **Train (80%)** | Model learns patterns here | Practice exams |
| **Test (20%)** | Model is scored here — never seen during training | Final exam |

**Why?** If we test on the same data we trained on, the model may have **memorized** answers instead of learning patterns.

`stratify=y` keeps Win/Draw/Loss proportions similar in both sets.

In [2]:
from sklearn.model_selection import train_test_split

# X = features (inputs), y = target (answers)
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

# Split: 80% train, 20% test; random_state=42 makes results repeatable
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training matches:", len(X_train))
print("Testing matches:", len(X_test))

Training matches: 4568
Testing matches: 1143


# Step 3 — Encoding + Pipeline

**Encoding** converts text (`"Brazil"`) into numbers for the model.

**OneHotEncoder** creates a 0/1 column for each category (team, tournament).

**Pipeline** chains steps: encode first → then train model. Professional teams always use pipelines.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Text columns need encoding; numeric columns pass through unchanged
text_columns = ["home_team", "away_team", "tournament"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            text_columns,
        )
    ],
    remainder="passthrough",  # keep is_neutral, year, month as-is
)

# Random Forest: learns rules from many decision trees (good first model for tabular data)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Pipeline = preprocessing + model in one object
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)

print("Pipeline ready:", pipeline)

Pipeline ready: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['home_team', 'away_team',
                                                   'tournament'])])),
                ('model', RandomForestClassifier(random_state=42))])


# Step 4 — Train the Model

**Training** = showing the model examples so it learns patterns.

`.fit(X_train, y_train)` = "learn from these features and these answers."

This is NOT magic — the model finds statistical patterns (e.g. certain team + tournament combos often win).

In [4]:
# Train on training data only (never touch X_test during fit)
pipeline.fit(X_train, y_train)

print("Training complete.")

Training complete.


# Step 5 — Evaluate on Test Data

**Accuracy** = how often the model is correct overall.

- Random guessing (3 classes) ≈ **33%**
- Our goal: beat 33% meaningfully

**Precision / Recall** (from the report):
- **Precision** — when model says "Win", how often is it right?
- **Recall** — of all actual Wins, how many did we catch?

First model with basic features may be ~50% — that is normal. We improve later with more data/features.

In [5]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on test set (model has never seen these matches)
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.1%}")
print(f"Random baseline (3 classes): ~33.3%")
print("\nDetailed report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 52.0%
Random baseline (3 classes): ~33.3%

Detailed report:
              precision    recall  f1-score   support

        Draw       0.29      0.13      0.18       262
        Loss       0.48      0.39      0.43       333
         Win       0.57      0.79      0.66       548

    accuracy                           0.52      1143
   macro avg       0.45      0.43      0.42      1143
weighted avg       0.48      0.52      0.48      1143



# Step 6 — Save the Model

**Saving** lets us reload the model later without retraining.

We use `joblib` — standard for scikit-learn models.

In [6]:
import joblib

# Create models folder if needed
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save the full pipeline (encoding + model together)
joblib.dump(pipeline, MODEL_PATH)

print("Model saved to:", MODEL_PATH.resolve())
print("File exists:", MODEL_PATH.exists())

Model saved to: C:\Users\HP-15\Desktop\worldcup_predictor\models\match_predictor.joblib
File exists: True


# Step 7 — Predict on a New Match (Inference)

**Inference** = using a trained model to predict something new.

We give it features for one hypothetical match — it returns Win, Draw, or Loss.

In [7]:
# Example: one new match (features only — no scores)
new_match = pd.DataFrame(
    [
        {
            "home_team": "Brazil",
            "away_team": "Argentina",
            "tournament": "Friendly",
            "is_neutral": 1,
            "year": 2024,
            "month": 6,
        }
    ]
)

# Load saved model (proves save/load works)
loaded_model = joblib.load(MODEL_PATH)

prediction = loaded_model.predict(new_match)[0]
probabilities = loaded_model.predict_proba(new_match)[0]
classes = loaded_model.named_steps["model"].classes_

print("Match: Brazil vs Argentina (Friendly, neutral venue)")
print("Predicted result for home team:", prediction)
print("\nProbabilities:")
for label, prob in zip(classes, probabilities):
    print(f"  {label}: {prob:.1%}")

Match: Brazil vs Argentina (Friendly, neutral venue)
Predicted result for home team: Win

Probabilities:
  Draw: 12.0%
  Loss: 36.0%
  Win: 52.0%


# Summary — You Built a Real ML Pipeline

| Step | Done |
|------|------|
| Load data | ✅ |
| Train/test split | ✅ |
| Encode text | ✅ |
| Train Random Forest | ✅ |
| Evaluate (~52% accuracy expected) | ✅ |
| Save model | ✅ |
| Predict new match | ✅ |

**Path A Phase 2 (later):** Add `goalscorers.csv` and more features to improve accuracy.

**Congratulations** — you completed the full ML pipeline once. 🎉